In [73]:
import pandas as pd
import geopandas as gpd
import json

In [53]:
df = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/new_pressure_time.csv", index_col=0)

In [3]:
df["datetime"] = pd.to_datetime(df["datetime"], format='%Y-%m-%d %H:%M:%S')

### Time segmentation: Morning, Afternoon, Evening and Night

In [4]:
# Define function to assign time segments
def assign_time_period(hour):
    if 7 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 23:
        return 'Evening'
    else:
        return 'Night'

# Apply function to dataframe to create new column 'time_period'
df['time_period'] = df['datetime'].dt.hour.apply(assign_time_period)

# Define function to assign day type (Weekday/Weekend)
def assign_day_type(day):
    if day < 5:
        return 'Weekday'
    else:
        return 'Weekend'

# Apply function to create new column 'day_type'
df['day_type'] = df['datetime'].dt.weekday.apply(assign_day_type)

In [23]:
df.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/pressure_segment_time.csv")

In [52]:
df

,Buurt,Wijk,place_id,reviews 2024,name,pressure,datetime,geometry,time_period,day_type
0,Felix Meritisbuurt,Grachtengordel-West,ChIJFTdsYMMJxkcRryoUM7EyJY0,133.0,Nooch,0.000000,2024-01-02 00:00:00,POINT (4.8840769 52.3723397),Night,Weekday
1,Felix Meritisbuurt,Grachtengordel-West,ChIJFTdsYMMJxkcRryoUM7EyJY0,133.0,Nooch,0.000000,2024-01-02 01:00:00,POINT (4.8840769 52.3723397),Night,Weekday
2,Felix Meritisbuurt,Grachtengordel-West,ChIJFTdsYMMJxkcRryoUM7EyJY0,133.0,Nooch,0.000000,2024-01-02 02:00:00,POINT (4.8840769 52.3723397),Night,Weekday
3,Felix Meritisbuurt,Grachtengordel-West,ChIJFTdsYMMJxkcRryoUM7EyJY0,133.0,Nooch,0.000000,2024-01-02 03:00:00,POINT (4.8840769 52.3723397),Night,Weekday
4,Felix Meritisbuurt,Grachtengordel-West,ChIJFTdsYMMJxkcRryoUM7EyJY0,133.0,Nooch,0.000000,2024-01-02 04:00:00,POINT (4.8840769 52.3723397),Night,Weekday
...,...,...,...,...,...,...,...,...,...,...
594403,Hoofdcentrum-Zuidoost,Amstel III/Bullewijk,ChIJJVNbSZcLxkcRmlFDljhOFgE,4724.0,Ziggo Dome,12.947432,2024-01-07 19:00:00,POINT (4.9371203 52.3135914),Evening,Weekend
594404,Hoofdcentrum-Zuidoost,Amstel III/Bullewijk,ChIJJVNbSZcLxkcRmlFDljhOFgE,4724.0,Ziggo Dome,25.894864,2024-01-07 20:00:00,POINT (4.9371203 52.3135914),Evening,Weekend
594405,Hoofdcentrum-Zuidoost,Amstel III/Bullewijk,ChIJJVNbSZcLxkcRmlFDljhOFgE,4724.0,Ziggo Dome,25.894864,2024-01-07 21:00:00,POINT (4.9371203 52.3135914),Evening,Weekend
594406,Hoofdcentrum-Zuidoost,Amstel III/Bullewijk,ChIJJVNbSZcLxkcRmlFDljhOFgE,4724.0,Ziggo Dome,25.894864,2024-01-07 22:00:00,POINT (4.9371203 52.3135914),Evening,Weekend


### Category Places

In [115]:
cat = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/places_categorised.csv", index_col=0)
cat = cat[["place_id", "name", "category"]]

In [116]:
cnt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/2024_count.csv")

In [117]:
cnt_cat = cnt.merge(cat, on = ["place_id", "name"], how = "left")
cnt_cat = cnt_cat.groupby(["place_id", "name", "category"]).sum(numeric_only = True).reset_index()

In [118]:
crds = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/times.csv")[["place_id", "coordinates"]]

# Parse the 'coordinates' column into separate latitude and longitude columns
crds[['latitude', 'longitude']] = crds['coordinates'].apply(
    lambda x: pd.Series(json.loads(x)))

crds = gpd.GeoDataFrame(
    crds,
    geometry=gpd.points_from_xy(crds['longitude'], crds['latitude']),
    crs="EPSG:4326")

crds = crds[["place_id", "geometry"]]

In [119]:
df_cat = cnt_cat.merge(crds, on = "place_id", how = "inner")
df_cat

,place_id,name,category,reviews 2024,rev_prc,geometry
0,ChIJ--17ntTjxUcR7UDCPpIqMWE,Enfes Restaurant,Dining & Cafes,64,0.003508,POINT (4.81065 52.35396)
1,ChIJ--5Ki5UJxkcRaVIVCb2RSX8,Kooi Coffeeshop,Dining & Cafes,187,0.010251,POINT (4.89943 52.36624)
2,ChIJ--rLyxIJxkcRcdcvOt9HCN4,Studio/K,Activities,306,0.016774,POINT (4.93603 52.36524)
3,ChIJ-17J8wPixUcRYhknGJNujXw,Dignita Vondelpark,Dining & Cafes,408,0.022365,POINT (4.85724 52.35184)
4,ChIJ-1X_9M0JxkcR6m3KYBdx9Xw,A Volo,Dining & Cafes,106,0.005810,POINT (4.8846 52.38428)
...,...,...,...,...,...,...
3013,ChIJzfXwHXQJxkcRfW99ZpqJKBQ,Caffe Milo,Dining & Cafes,144,0.007893,POINT (4.92551 52.36025)
3014,ChIJzfayl0QJxkcRrDpFzfgA4ac,Clos Amsterdam,Dining & Cafes,170,0.009319,POINT (4.91808 52.35671)
3015,ChIJzfy_7V0LxkcR_OA8F3r9-as,Parade Amsterdam,Event venue,65,0.003563,POINT (4.91002 52.34054)
3016,ChIJzwzS5owJxkcRDZfCjur-OPU,Sinne,Dining & Cafes,97,0.005317,POINT (4.89407 52.353)


In [120]:
df_cat.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/pressure_segment_category.csv")

### Capacity based on RUDIFUN